# TeAAL specifications on variants of Loops' SpMM implementations (Work Oriented)

Description: All threads are assigned (`TOTAL_WORK` / `Number of Processors`) work items. Each thread then sequentially processes assigned work items in a loop. `TOTAL_WORK` = `NNZ` + `NUM_ROWS`.

Template: DNE in Loops

Scheduler: https://github.com/gunrock/loops/blob/main/include/loops/schedule/work_oriented.hxx

## Imports

Import the necessary modules.

In [1]:
# HiFiber boilerplate

from fibertree_bootstrap import *

fibertree_bootstrap(style="tree", animation='movie')

# Compilation boilerplate

import os
import sys
sys.path.insert(0, "../..")

from src import utils

Running bootstrap
The fibertree module is already installed and available to import


interactive(children=(Dropdown(description='style', options=('tree', 'uncompressed', 'tree+uncompressed'), val…

Button(description='Run all cells below', style=ButtonStyle())

## Initialization

Initialize the input tensors.

For simplicity, the size of a thread warp is the same as the size of a thread block (`WARP_SIZE = BLOCK_SIZE`). Suppose that each GPU SM processes 1 thread warp/block per cycle.

In [2]:
K = 4
M = 4
N = 4

# GPU Kernel Configuration
BLOCK_SIZE = 2 # Number of threads per block
GRID_SIZE = 2 # Number of thread blocks

print(f"GPU Kernel Configuration\n \
        BLOCK_SIZE (Number of threads per block): {BLOCK_SIZE} \n \
        GRID_SIZE (Number of thread blocks): {GRID_SIZE}")

seed = 1

#A_MK = Tensor.fromRandom(rank_ids=["M", "K"], shape=[M, K], seed=seed, density=[0.9, 0.6], name="A")
A_MK = Tensor.fromUncompressed(rank_ids=["M", "K"], root=[[0,0,8,0],[4,1,0,10],[8,4,2,0],[9,7,5,0]], shape=[M, K], name="A")
B_KN = Tensor.fromRandom(rank_ids=["K", "N"], shape=[K, N], seed=seed + 1, density=[1, 1], name="B") # SpMM, B is dense

# Calculating A_NNZ
A_NNZ = 0

for row in A_MK.getRoot().getPayloads():
    for nz in row.getPayloads():
        A_NNZ += 1

print(f"A_NNZ: {A_NNZ}")

TOTAL_WORK = A_NNZ
WORK_PER_THREAD = math.ceil(TOTAL_WORK / (BLOCK_SIZE * GRID_SIZE))
print(f"TOTAL_WORK: {TOTAL_WORK}, WORK_PER_THREAD = {WORK_PER_THREAD}")

GPU Kernel Configuration
         BLOCK_SIZE (Number of threads per block): 2 
         GRID_SIZE (Number of thread blocks): 2
A_NNZ: 10
TOTAL_WORK: 10, WORK_PER_THREAD = 3


## TeAAL Specifications

Rows of matrix $A$ are partitioned across the SMs' warp/block. A thread warp/block can be assigned to a row with all zeros. 

Note that the current TeAAL specificaiton does not allow to specify the rank of `opt: slip`. This means there exists a synchronization across the SMs.

In [3]:
yaml = """
einsum:
  declaration:
    A: [M, K]
    B: [K, N]
    Z: [M, N]
  expressions:
    - Z[m, n] = A[m, k] * B[k, n]
mapping:
  rank-order:
    A: [M, K]
    B: [K, N]
    Z: [M, N]
  partitioning:
    Z:
      (M, K): [flatten()]
      MK: [uniform_occupancy(A.WORK_PER_THREAD)]
  loop-order:
    Z: [MK1, MK0, N]
    # MK1: Number of iterations to process all NNZ in parallel
    # MK0: Number of NNZ assigned to each thread = WORK_PER_THREAD
  spacetime:
    Z:
      space: [MK1]
      time: [MK0, N]
"""

utils.compile(yaml)

In [4]:
# Autogenerated HiFiber

Z_MN = Tensor(rank_ids=["M", "N"], name="Z")
z_m = Z_MN.getRoot()
tmp0 = A_MK
tmp1 = tmp0.flattenRanks(depth=0, levels=1, coord_style="tuple")
A_MK_flat = tmp1
A_MK_flat.setRankIds(rank_ids=["MK"])
b_k = B_KN.getRoot()
a_mk = A_MK_flat.getRoot()
canvas = createCanvas(A_MK_flat, B_KN, Z_MN)
A_MK_flat = Tensor.fromFiber(rank_ids=["MK"], fiber=a_mk, name="A")
tmp2 = A_MK_flat
tmp3 = tmp2.splitEqual(WORK_PER_THREAD)
A_MK1MK0 = tmp3
A_MK1MK0.setRankIds(rank_ids=["MK1", "MK0"])
a_mk1 = A_MK1MK0.getRoot()
for mk1_pos, (mk1, a_mk0) in enumerate(a_mk1):
    for mk0_pos, ((m, k), a_val) in enumerate(a_mk0):
        z_n = z_m.getPayloadRef(m)
        b_n = b_k.getPayload(k)
        for n_pos, (n, (z_ref, b_val)) in enumerate(z_n << b_n):
            z_ref += a_val * b_val
            canvas.addActivity(((m, k),), (k, n), (m, n), spacetime=((mk1_pos,), (mk0_pos, n_pos)))
displayCanvas(canvas)

Starting simulation
Finished simulation


Create individual tensor images for each cycle:   0%|          | 0/12 [00:00<?, ?it/s]

Paste individual tensor images into frame for each cycle:   0%|          | 0/14 [00:00<?, ?it/s]

Render video frame for each cycle:   0%|          | 0/14 [00:00<?, ?it/s]

## Check Results

Check that generated code computes the correct result.

**Note**: Should be used after compiling and running the kernel (above cell).

In [ ]:
utils.check_matmul(A_MK, B_KN, Z_MN)

## Performance on GPU

Load Balance: Better load balance than Thread-Mapped, since rows with high NNZ will be processed by a group of threads (smoothing out heavy rows). Additionally, there's no need to worry about load balance across the warps since SMs can simply start processing on another thread warp when one warp finishes earlier than the others.

Assuming that the $A$, $B$, and $Z$ are stored in CSR format, the memory access pattern would be:
- A: Coalesced access, threads in a warp are accessing the same row of $A$.
- B: Uncoalesced access, threads are accessing $B$ in column-wise order and at random positions based on column indices of $A$'s nonzero entries.
- Z: Coalesced access, threads in a warp are writing the same row of $Z$. At least with writing to $Z$, it may not be as fast as Thread-Mapped since it is done atomically on Group-Mapped to avoid data conflict. 